# Import Dependencies

In [ ]:
import pymupdf as pmp
import re

print(pmp.__doc__)

In [ ]:
%pwd

In [ ]:
import os
os.chdir("./..")
%pwd

In [ ]:
#docs = pmp.open("/home/jovyan/work/EU_MDR_RAG/data/raw/eu_mdr_2017-745.pdf")
docs = pmp.open("data/raw/eu_mdr_2017-745.pdf")
print(len(docs))

In [ ]:
doc = pmp.open("data/raw/eu_mdr_2017-745.pdf")
page_starts = []
data=""
for num,page in enumerate(doc):
    page_starts.append(len(data))
    if num >= 173:
        break
    text = str(page.get_text())
    data += text

### Observation:
1. Here, in the regulations the new line starts with Article 8, which pattern matches and creates a new document. The issue is it is not new article and will lead to poor results.
2. Also, i altered the pattern because the regex matches with only one line so i need to either find the all the patterns and then extract the text between them or i have to create two line lookahead function.

In [ ]:
import re
pattern_chapter = "^(CHAPTER [IVX]+) \n(.+)"
s = re.finditer(pattern_chapter, data, re.M)
for m in s:
    print(m.group())

### Idea:
The plan is to collect all the title's location and extract the data between them.

In [ ]:
pattern_annex = "^(ANNEX [IVX]+) \n(.+)"
pattern_chapter = "^(CHAPTER [IVX]+) \n(.+)"
pattern_section = "^(SECTION [0-9]+) \n(.+)"
pattern_article ="^(Article [0-9]+) \n(?!Article)(?!— )(.+)"
markers = list()
for pattern in [pattern_annex, pattern_article, pattern_chapter, pattern_section]:
    iter_obj = re.finditer(pattern, data, re.M)
    for match in iter_obj:
        if pattern == pattern_annex:
            markers.append((match.start(),match.end(), "Annex", match.group()))
        if pattern == pattern_chapter:
            markers.append((match.start(),match.end(), "Chapter", match.group()))
        if pattern == pattern_article:
            markers.append((match.start(),match.end(), "Article", match.group()))
        if pattern == pattern_section:
            markers.append((match.start(),match.end(), "Section", match.group()))
sorted_markers=sorted(markers)
sorted_markers

In [ ]:
sorted_markers[0][0]

In [ ]:
import bisect
p = bisect.bisect(page_starts,sorted_markers[0][0])
q = bisect.bisect(page_starts,sorted_markers[1][0])
p,q

In [ ]:
import bisect
Documents = []
current_text = ""
chapter = ""
section = ""
section_header = ""
start_location = 0
end_location = sorted_markers[-1][1]
metadata = {"document_name": "eu_mdr_2017-745.pdf", "document_type": "Regulations", "chapter": "", "chapter_title": "", "article": "", "article_title": "", "annex": "","annex_title": "", "section": "", "section_title": "", "page_number": "", "cross_references": ""}

pc = data[0:sorted_markers[0][1]]
mc = metadata.copy()
mc['chapter'] = 0
mc['chapter_title'] = 'preamble'
mc['page_number'] = [bisect.bisect(page_starts, 0), bisect.bisect(page_starts,sorted_markers[0][0])]
doc_1 =Documents.append({"page_content": pc, "metadata": mc})

for i, marker in enumerate(sorted_markers):
    if i < len(sorted_markers) - 1:
        current_text = data[marker[0]:sorted_markers[i+1][0]]
    else:
        current_text = data[marker[1]:]
    new_metadata = metadata.copy()

    if marker[2] == "Chapter":
        chapter = marker[3].split("\n")[0]
        chapter_title = marker[3].split("\n")[1]
        new_metadata['chapter'] = chapter
        new_metadata['chapter_title'] = chapter_title
        ch_location = marker[0]
        next_ch_loc = sorted_markers[i+1][0]
        page_num = bisect.bisect(page_starts,ch_location)
        end_pnum = bisect.bisect(page_starts, next_ch_loc)
        new_metadata['page_number'] = f"{page_num}-{end_pnum}"
        metadata.update(new_metadata)
        Documents.append({"page_content": current_text, "metadata": new_metadata})

    elif marker[2] == "Section":
        section = marker[3].split("\n")[0]
        section_title = marker[3].split("\n")[1]
        new_metadata['section'] = section
        new_metadata['section_title'] = section_title
        sec_location = marker[0]
        next_ch_loc = sorted_markers[i+1][0]
        page_num = bisect.bisect(page_starts, sec_location)
        end_pnum = bisect.bisect(page_starts, next_ch_loc)
        new_metadata['page_number'] = f"{page_num}-{end_pnum}"
        metadata.update(new_metadata)
        Documents.append({"page_content": current_text, "metadata": new_metadata})
    
    elif marker[2] == "Article":
        article = marker[3].split("\n")[0]
        article_title = marker[3].split("\n")[1]
        new_metadata['article'] = article
        new_metadata['article_title'] = article_title
        art_location = marker[0]
        page_num = bisect.bisect(page_starts,art_location)
        next_ch_loc = sorted_markers[i+1][0]
        end_pnum = bisect.bisect(page_starts, next_ch_loc)
        new_metadata['page_number'] = f"{page_num}-{end_pnum}"
        metadata.update(new_metadata)
        Documents.append({"page_content": current_text, "metadata": new_metadata})

    
    elif marker[2] == "Annex":
        annex = marker[3].split("\n")[0]
        annex_title = marker[3].split("\n")[1]
        new_metadata['annex'] = annex
        new_metadata['annex_title'] = annex_title
        ann_location = marker[0]
        page_num = bisect.bisect(page_starts,ann_location)
        if i < len(sorted_markers) - 1:
            next_ch_loc = sorted_markers[i+1][0]
        end_pnum = bisect.bisect(page_starts, next_ch_loc)
        new_metadata['page_number'] = f"{page_num}-{end_pnum}"
        metadata.update(new_metadata)
        Documents.append({"page_content": current_text, "metadata": new_metadata})
        metadata['chapter'] = ""
        metadata['chapter_title'] = ""
        metadata['article'] = ""
        metadata['article_title'] = ""
        metadata['section'] = ""
        metadata['section_title'] = ""
            
    #Documents.append({"page_content": current_text, "metadata": metadata})

In [ ]:
len(Documents)

In [ ]:
Documents

Observation:
1. Now, all the data extratced and created as document with metadata.
2. I still need to do data cleaning like removing footers and so.
3. I observed the pattern by checking the documents using pymupdf.open. I checked first three pages and some random pages how the headers and footers are represented.
4. The pattern is simillar almost at every page. Theheader stats with:

`" \n5.5.2017 \nL 117/1 \n.............."
`
The footer starts in the same way butwith extra spaces like shown below:

`"     \n(1) Regulation (EC) No 178/2002 of the European Parliament and of"`

almost at every page.

* But first remove the uncode characters for consistency.

Hint:
`ord()` function

In [ ]:
special_chr = []
d = {}
for i in data:
    p = ord(i)
    if p > 127:
        special_chr.append(chr(p))
        d[p]=chr(p)

print(set(special_chr),d)

In [ ]:
data[data.find("Ε")-50: data.find("Ε")+50]

In [ ]:
data[data.find("à")-50: data.find("à")+50]

In [ ]:
data.find("\xad")

In [ ]:
data[data.find("\xad")-50: data.find("\xad")+50]

In [ ]:
print(data.count("à"), data.count("\xad"), data.count("‑"), data.count('‘'), data.count('’'), data.count('Ε'))

In [ ]:
def clean(corpus):

    corpus = corpus.replace("\xad","")
    corpus = corpus.replace("Ε","E")
    corpus = corpus.replace("‑","-")
    corpus = corpus.replace("‘","'")
    corpus = corpus.replace("’","'")

    return corpus

new_data = clean(data)

In [ ]:
print(new_data.count("à"), new_data.count("\xad"), new_data.count("‑"), new_data.count('‘'), new_data.count('’'), new_data.count('Ε'))

In [ ]:
new_data[4250:4350]

### Now, removeing Header/footers

In [ ]:
header_pattern = r"""\d{1,2}\.\d{1,2}\.\d{4}\s*\nL\s+\d{3}/\d+\s*\nOfficial Journal of the European Union\s*\nEN\s*"""
re.findall(header_pattern, new_data)

Observation:
1. One thing to see is the footer contains the cross reference with some stated papers as follows:

`Opinion of 14 February 2013 (OJ C 133, 9.5.2013, p. 52).`

for moment i am planning to keep it. if in future i make an agent for it then it might help to find the relation.

In [ ]:
# Remove Headers

def remove_header(corpus):
    new_data=re.sub(header_pattern,"",corpus)